# Chapter 0: Mathematical and Physical Preliminaries

This solved notebook is the executable companion to the new foundations chapter.
It starts with hand-sized numbers and then repeats the same objects in NumPy and
PyTorch.

Progression:

1. Scalars, vectors, matrices, norms, and path sums.
2. Scalar residuals, fixed points, and damping.
3. Finite differences and exact Jacobians.
4. JVP/VJP checks on a small PyTorch function.
5. Graph locality and mean-field global context derived from symmetry.
6. Hutchinson probes, implicit adjoints, and physical relaxation language.

The point is not speed. The point is that every later DEQ, MDEQ, SILVA, PDE,
distributional, or physics chapter uses these same objects.


In [ ]:

import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "silva_networks").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not find suite root containing src/silva_networks")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt
from silva_networks import (
    np_picard,
    np_finite_difference_jacobian,
    np_exact_tanh_affine_jacobian,
    np_power_iteration,
    np_implicit_gradient,
    picard,
    anderson,
    torch_full_jacobian,
    torch_vjp,
    torch_jvp,
    TinySILVALayer,
    spectral_radius_vjp,
    hutchinson_jacobian_frobenius,
)

np.random.seed(7)
torch.manual_seed(7)
torch.set_default_dtype(torch.float64)


## 1. Scalars, vectors, matrices, norms, and path sums


In [ ]:

# 1. Linear algebra with hand-sized numbers.
A = np.array([[1.0, 2.0],
              [3.0, 4.0]])
z = np.array([2.0, -1.0])

Az_by_hand = np.array([
    1.0 * 2.0 + 2.0 * (-1.0),
    3.0 * 2.0 + 4.0 * (-1.0),
])
Az_numpy = A @ z
print("Az by hand:", Az_by_hand)
print("Az by NumPy:", Az_numpy)

v = np.array([3.0, -4.0])
M = np.array([[1.0, 2.0],
              [0.0, 2.0]])
print("||v||_2:", np.sqrt(np.sum(v ** 2)))
print("||M||_F:", np.sqrt(np.sum(M ** 2)))

# Matrix powers and a finite path sum.
T = np.array([[0.2, 0.0],
              [0.0, 0.5]])
s = np.array([1.0, 2.0])
terms = []
for t in range(5):
    term = np.linalg.matrix_power(T, t) @ s
    terms.append(term)
    print(f"T^{t} s =", term)
z5 = sum(terms)
z_inf = np.linalg.solve(np.eye(2) - T, s)
print("finite five-term path sum:", z5)
print("infinite response (I-T)^-1 s:", z_inf)


## 2. Scalar residuals, fixed points, and damping


In [ ]:

# 2. Scalar fixed points, residuals, and damping.
def f_scalar(z):
    return 0.5 * z + 2.0

# Solve z = 0.5 z + 2 by moving all z terms to the left:
# z - 0.5 z = 2, so 0.5 z = 2, so z = 4.
z_star_exact = 4.0
for test_z in [0.0, 3.0, 4.0]:
    residual_value = f_scalar(test_z) - test_z
    print("z =", test_z, "f(z)-z =", residual_value)

alpha = 0.25
z_current = 0.0
trajectory = [z_current]
for k in range(8):
    proposal = f_scalar(z_current)
    displacement = proposal - z_current
    z_current = z_current + alpha * displacement
    trajectory.append(z_current)
    print(f"k={k+1}: proposal={proposal:.6f}, displacement={displacement:.6f}, z={z_current:.6f}")

plt.plot(trajectory, marker="o")
plt.axhline(z_star_exact, linestyle="--", color="black", label="exact fixed point")
plt.xlabel("iteration")
plt.ylabel("z")
plt.title("Damped scalar relaxation")
plt.legend()
plt.show()


## 3. Finite differences and exact Jacobians


In [ ]:

# 3. Finite differences and exact Jacobian for f(z)=tanh(Wz+s).
W_small = np.array([[0.2, -0.1],
                    [0.05, 0.25]])
s_small = np.array([0.4, -0.3])
z0_small = np.zeros(2)

def f_small_np(z):
    return np.tanh(W_small @ z + s_small)

J_fd = np_finite_difference_jacobian(f_small_np, z0_small, eps=1e-5)
J_exact = np_exact_tanh_affine_jacobian(W_small, z0_small, s_small)

print("preactivation Wz+s:", W_small @ z0_small + s_small)
print("diag entries 1-tanh(pre)^2:", 1.0 - np.tanh(s_small) ** 2)
print("finite-difference Jacobian:\n", J_fd)
print("exact Jacobian:\n", J_exact)
print("agreement ||J_fd-J_exact||_F:", np.linalg.norm(J_fd - J_exact))

# Show one central-difference column explicitly.
eps = 1e-3
e1 = np.array([1.0, 0.0])
column_1 = (f_small_np(z0_small + eps * e1) - f_small_np(z0_small - eps * e1)) / (2 * eps)
print("first Jacobian column from central difference:", column_1)
print("first exact column:", J_exact[:, 0])


## 4. JVP and VJP in PyTorch


In [ ]:

# 4. PyTorch JVP and VJP on the same small map.
W_t = torch.tensor(W_small)
s_t = torch.tensor(s_small)
z_t = torch.zeros(2, requires_grad=True)

def f_small_torch(z):
    return torch.tanh(W_t @ z + s_t)

J_full = torch_full_jacobian(f_small_torch, z_t.detach())
v_t = torch.tensor([1.0, -2.0])
_, Jv = torch_jvp(f_small_torch, z_t.detach(), v_t)
JTv = torch_vjp(f_small_torch, z_t.detach(), v_t)

print("full Jacobian:\n", J_full.detach().numpy())
print("J @ v from materialized J:", (J_full @ v_t).detach().numpy())
print("JVP from autograd:", Jv.detach().numpy())
print("J.T @ v from materialized J:", (J_full.T @ v_t).detach().numpy())
print("VJP from autograd:", JTv.detach().numpy())


## 5. Graph locality and mean-field global context


In [ ]:

# 5. Local graph aggregation and mean-field global context.
H = np.array([[2.0],
              [5.0]])
A_graph = np.array([[0.0, 1.0],
                    [1.0, 0.0]])
print("AH swaps the two node states:\n", A_graph @ H)

Y = np.array([[1.0, 0.0],
              [3.0, 2.0],
              [5.0, 4.0]])
Wg = np.array([[1.0, -1.0],
               [0.5, 0.25]])

# Start from the empirical mean definition.
mean_loop = np.zeros(Y.shape[1])
for j in range(Y.shape[0]):
    mean_loop = mean_loop + Y[j]
mean_loop = mean_loop / Y.shape[0]
print("mean from explicit sum:", mean_loop)

# Broadcast and mix.
global_loop = np.zeros((Y.shape[0], Wg.shape[0]))
for i in range(Y.shape[0]):
    global_loop[i] = mean_loop @ Wg.T

one = np.ones((Y.shape[0], 1))
P_global = one @ one.T / Y.shape[0]
global_matrix = P_global @ Y @ Wg.T
print("global term from loops:\n", global_loop)
print("global term from matrix formula:\n", global_matrix)
print("loop/matrix difference:", np.linalg.norm(global_loop - global_matrix))

# Permutation equivariance check: G(Pi Y) = Pi G(Y).
perm = np.array([2, 0, 1])
Pi = np.eye(3)[perm]
lhs = P_global @ (Pi @ Y) @ Wg.T
rhs = Pi @ (P_global @ Y @ Wg.T)
print("permutation equivariance error:", np.linalg.norm(lhs - rhs))


## 6. Probability probes and physical relaxation


In [ ]:

# 6. Hutchinson probe and physical relaxation.
J = np.array([[1.0, 2.0],
              [0.0, 1.0]])
v_probe = np.array([1.0, -1.0])
Jv = J @ v_probe
print("Jv:", Jv)
print("single Hutchinson probe ||Jv||^2:", float(Jv @ Jv))
print("exact ||J||_F^2:", float(np.sum(J ** 2)))

rng = np.random.default_rng(7)
estimates = []
for _ in range(1000):
    v = rng.choice([-1.0, 1.0], size=2)
    estimates.append(np.sum((J @ v) ** 2))
print("average over 1000 Rademacher probes:", float(np.mean(estimates)))

# Physical relaxation: z_{k+1}=z_k+alpha(f(z_k)-z_k).
def f_relax(z):
    return np.array([0.5 * z[0] + 1.0, 0.25 * z[1] - 1.0])

z = np.zeros(2)
alpha = 0.4
residuals = []
for _ in range(25):
    field = f_relax(z) - z
    residuals.append(np.linalg.norm(field))
    z = z + alpha * field
print("relaxed state:", z)
print("last vector-field norm:", residuals[-1])
plt.semilogy(residuals, marker="o")
plt.xlabel("iteration")
plt.ylabel("||f(z)-z||")
plt.title("Residual as vector-field magnitude")
plt.show()


## 7. Implicit adjoint solve


In [ ]:

# 7. One adjoint solve by hand and by NumPy.
J = np.array([[0.2, 0.1],
              [0.0, 0.3]])
g = np.array([1.0, 2.0])
A_adj = np.eye(2) - J.T
lam = np.linalg.solve(A_adj, g)

print("I-J.T:\n", A_adj)
print("lambda:", lam)
print("check (I-J.T) lambda:", A_adj @ lam)

# If df/dtheta has two parameter columns, lambda^T df/dtheta gives the gradient.
df_dtheta = np.array([[1.0, 0.0],
                      [2.0, -1.0]])
grad_theta = lam @ df_dtheta
print("lambda^T df/dtheta:", grad_theta)


## 8. Exercise-linked checks


In [ ]:

# 8. Tiny checks tied to Chapter 0 exercises.
print("P0.E3 fixed point for f(z)=0.25z+3:", 3.0 / (1.0 - 0.25))

def square(z):
    return z ** 2
eps = 0.1
central = (square(2.0 + eps) - square(2.0 - eps)) / (2 * eps)
print("P0.E4 central difference for z^2 at z=2:", central)

T = np.array([[0.2, 0.0],
              [0.0, 0.5]])
s = np.array([1.0, 2.0])
print("P0.M3 (I-T)^-1 s:", np.linalg.solve(np.eye(2) - T, s))

# Mean-field Jacobian action:
# If G(Y)=P Y Wg.T, then dG[V]=P V Wg.T.
V = np.random.randn(*Y.shape)
dG = P_global @ V @ Wg.T
eps = 1e-6
finite_dG = (P_global @ (Y + eps * V) @ Wg.T - P_global @ Y @ Wg.T) / eps
print("mean-field Jacobian action agreement:", np.linalg.norm(dG - finite_dG))


## Chapter 0 executable summary

Chapter 0 is the math-and-physics foundation in executable form. Every later chapter specializes these same ideas to DEQ, MDEQ, SILVA, PDE, graph, distributional, or algorithmic equilibrium settings.
